# Chapter 10 Companion Notebook: Tree-Based Models: Decision Tree Classification

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch10_Tree_Based_Models_Decision_Tree.ipynb)

This notebook accompanies Chapter 10 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
- Click "Upload" and select this file and the data file.

# Bank customers: Decision trees

### Use "Bank customer.csv"
- age (numeric)  
- marital: marital status (categorical: "married", "divorced", "single"; "divorced" means divorced or widowed)  - education (categorical: "secondary", "primary", "tertiary")  
- default: has credit in default? (binary: "yes", "no")  
- balance: average yearly balance, in euros (numeric)  
- housing: has housing loan? (binary: "yes", "no")  
- loan: has personal loan? (binary: "yes", "no")  
- duration: last contact duration, in seconds (numeric)  
- campaign: number of contacts performed during this campaign (numeric, includes last contact)  
- pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric)
- previous: number of contacts performed before this campaign (numeric)  
- poutcome: outcome of the previous marketing campaign (categorical: "failure", "success")
- deposit: has the client subscribed a term deposit? (binary: "yes", "no")

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt

In [ ]:
# Read the data

df = pd.read_csv('Bank customer.csv')
df.head()

In [ ]:
# Create dummy variables for categorical columns

df = pd.get_dummies(df, drop_first=True)
df.head()

In [ ]:
# Define x and y. Split into train, test data

y=df.deposit_yes
x=df[['age', 'balance', 'duration', 'campaign']]
xtrain, xtest, ytrain, ytest = train_test_split(x, y, random_state=1)

### Run a decision tree

In [ ]:
dt = DecisionTreeClassifier(criterion="entropy", max_depth=4).fit(xtrain, ytrain)
pred = dt.predict(xtest)  # Prediction

# Predict the target in the test data and display accuracy, confusion matrix
print("Accuracy: ", metrics.accuracy_score(ytest, pred))
print(confusion_matrix(ytest,pred))

|          | Predicted 0 | Predicted 1 |
|----------|-------------|-------------|
| Actual 0 |      TN     |      FP     |
| Actual 1 |      FN     |      TP     |

- `m.predict(xtest)`: Trained model m is used to predict the target variable on the test data.
- `metric.accuracy_score`: calculates the accuracy of the predicted compared to the actual target values (ytest).
- `confusion_matrix(ytest, pred)`: generates a matrix showing the count of true positive, true negative, false positive, and false negative predictions, helping to assess the performance of a classification model.
  - true positive: correctly predicted as positive by a classification model.
  - true negative: correctly predicted as negative by a classification model.
  - false positives: instances incorrectly predicted as positive.
  - false negative: instances incorrectly predicted as negative.

In [ ]:
### Classification report

print("Decision Tree Accuracy", accuracy_score(ytest, pred))
print("Decision Tree Confusion Matrix \n", confusion_matrix(ytest, pred))
print("Decision Tree Classification Report \n", classification_report(ytest, pred))

- `DecisionTreeClassifier()` initializes a Decision Tree classifier object.
  - `fit(xtrain, ytrain)` This trains the Decision Tree classifier on the training data `xtrain` (features) and `ytrain` (target).
- `pred1 = dt.predict(xtest)` generates predictions (`pred1`) on the test data (`xtest`) using the trained Decision Tree classifier (`dt`).
- `accuracy_score(ytest, pred1))` calculates the accuracy of the Decision Tree classifier by comparing the predicted values (`pred1`) with the actual target values (`ytest`) using the `accuracy_score()` function .
- `confusion_matrix(ytest, pred1))` computes the confusion matrix for the Decision Tree classifier.
- The confusion matrix provides a summary of the classifier's performance by showing the counts of true positives, true negatives, false positives, and false negatives.
- `classification_report(ytest, pred1))` generates a classification report for the Decision Tree classifier.
- `\n` is a newline character.

### Visualization

In [ ]:
from sklearn.tree import export_graphviz
import pydotplus
from IPython.display import Image

# Export the decision tree to DOT format
dot_data = export_graphviz(dt, out_file=None, feature_names=xtrain.columns, class_names=["No", "Yes"],
                      filled=True, rounded=True, special_characters=True)

# Convert the DOT file to a graph
graph = pydotplus.graph_from_dot_data(dot_data)

# Display the decision tree as an image
Image(graph.create_png())

### Interpretation of the decision tree
1. Root Node: The first split is based on "age ≤ 59.5." If True (age ≤ 59.5), the samples go to the left child node. If False (age > 59.5), the samples go to the right child node.
2. Left Branch (age ≤ 59.5): The next split is "balance ≤ 908.0." This separates customers with lower balances from those with higher balances.Additional splits continue with features like "campaign" (number of contacts during the campaign) and "balance," refining the classification further.
3. Right Branch (age > 59.5): The first split is "campaign ≤ 1.5." Customers contacted fewer times are evaluated. Further splits analyze features like "balance" and "age" to predict the likelihood of deposit.
4. Terminal Nodes (Leaf Nodes): Leaf nodes represent the final prediction. For example, a node with "entropy = 0.0, value = [10, 0]" predicts all No. A node with "value = [5, 16]" predicts Yes because most samples belong to the Yes class.
- The value represents the count of samples in each class ([No, Yes]).

### Insights from the Tree:
1. Key Features: Features like "age," "balance," and "campaign" play a critical role in determining deposit likelihood. For instance, younger customers (age ≤ 59.5) with lower balances are more likely to fall in the No class.
2. Class Separation: High purity nodes (low entropy) indicate effective splits. For example, the node with "entropy = 0.0" is highly confident in predicting the No class.
3. Imbalance: Some splits create imbalanced nodes (e.g., "value = [5, 16]"), but they still provide meaningful predictions based on the majority class.

### Hyperparameter Tuning
Parameter examples in a decision tree: `help(DecisionTreeClassifier)`
- **criterion**: Gini(default) or entropy. It defines the function to measure the quality of a split.
- **max_depth**: The maximum depth of the tree. It can take any integer value or None(default). If None, nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.
- **min_samples_leaf**: The minimum number of samples required to be at a leaf node. Integer or float(percentage). Default=1
- **min_samples_split**: The minimum number of samples required to split an internal node. Integer or float(percentage). Default=2
- **max_features**: the number of features to consider when looking for the best split.
- **max_leaf_nodes**: the maximum number of possible leaf nodes. If None(default), then it takes an unlimited number of leaf nodes.
- **min_impurity_split**: the threshold for early stopping tree growth. A node will split if its impurity is above the threshold otherwise it is a leaf.

#### Tuning `max_depth`

In [ ]:
# GridSearchCV to find optimal max_depth

param={'max_depth': range(1,21)}
m = DecisionTreeClassifier(criterion="entropy", random_state=10)
tr = GridSearchCV(m, param, cv=5, scoring="accuracy")
tr.fit(xtrain, ytrain)

- `DecisionTreeclassifier`
  - `criterion="entropy"`: To see the information gain of each node
  - Entropy measures the impurity or disorder. It is used to evaluate the quality of a split in a decision tree.
- `GridSearchCV(m,param,cv=5,scoring="accuracy")`: A grid search scheme consists of
    - an estimator (classifier such as SVC() or decision tree): m
    - a parameter space: param={'max_depth': range(1,21)} (Values from 1 to 20 is considered for the max_depth)
        range(start, end, step): start is inclusive, stop is exclusive, step is optional
    - a cross-validation scheme: cv=5 (the dataset will be split into 5 parts, and the model will be trained and evaluated 5 times, each time using a different part as the validation set.)
    - a score function: accuracy
    - a method for searching or sampling candidates (optional)

In [ ]:
# GridSearch CV scores (accuracies)

accu = tr.cv_results_
pd.DataFrame(accu).head()

- `accu = tr.cv_results_`: takes the result dictionary from a cross-validation search (stored in tr) and assigns it to the variable accu.
- `pd.DataFrame(accu)`: Creating a DataFrame using the accu dictionary. The DataFrame will have columns corresponding to different pieces of information collected during the grid search, such as mean fit times, mean test scores, etc.

In [ ]:
# Visualize accuracy rates with max_depth

plt.plot(accu["param_max_depth"], accu["mean_test_score"], marker='o')
plt.xlabel("Max Depth")
plt.ylabel("Accuracy")

- As we increase max_depth, the accuracy increases until max-depth=3, after which the accuracy declines
- Accuracies are average accuracies across the 5-folds.

#### Tuning `min_samples_leaf`
- `range(5,201,20)`: Values from 5 to 200 with a step of 20 is considered for `min_samples_leaf`.

In [ ]:
# GridSearchCV to find optimal min_samples_leaf

param={'min_samples_leaf': range(5,201,20)}
m = DecisionTreeClassifier(criterion = "entropy", random_state=10)
tr = GridSearchCV(m, param, cv=5, scoring="accuracy")
tr.fit(xtrain, ytrain)

In [ ]:
# accuracies of GridSearch CV

accu = tr.cv_results_
pd.DataFrame(accu).head(3)

In [ ]:
# Plot accuracies with min_samples_leaf

plt.plot(accu["param_min_samples_leaf"], accu["mean_test_score"], marker='o')
plt.xlabel("Min sample leaf")
plt.ylabel("Accuracy")

- As we increase min_samples_leaf, the accuracy increases until min_samples_leaf=125, after which the accuracy declines.
- Accuracies are average accuracies across the 5-folds.

#### Tuning `min_samples_split`
- `range(5,301,25)`: Values from 5 to 300 with a step of 25 is considered for `min_samples_split`.

In [ ]:
# GridSearchCV to find optimal min_samples_split

param={'min_samples_split': range(5,301,25)}
m = DecisionTreeClassifier(criterion = "entropy", random_state = 10)
tr = GridSearchCV(m, param, cv=5, scoring="accuracy")
tr.fit(xtrain, ytrain)

In [ ]:
# scores of GridSearch CV

accu = tr.cv_results_
pd.DataFrame(accu).head()

In [ ]:
# plot accuracies with min_samples_split

plt.plot(accu["param_min_samples_split"], accu["mean_test_score"], marker="o")
plt.xlabel("Min samples split")
plt.ylabel("Accuracy")

- As we increase `min_samples_split`, the accuracy increases until `min_samples_split`=250, after which the accuracy stabilizes or slightly declines.
- Accuracies are average accuracies across the 5-folds.

<hr>

## Grid search to find optimal hyperparameters together.

In [ ]:
# Create the parameter grid and fit the model

param = {'max_depth': range(1,10,3), 'min_samples_leaf': range(50,151,25),
         'min_samples_split': range(50,151,30), 'criterion': ["entropy"]}
m = DecisionTreeClassifier()
grid_search = GridSearchCV(estimator=m, param_grid=param, cv=5, verbose = 1)
grid_search.fit(xtrain, ytrain)

- `param`: defines the hyperparameter space to search over during grid search.
  - `max_depth`: Values from 5 to 15, incrementing by 5 at each step.
  - `min_samples_leaf`: Values from 50 to 150, incrementing by 30 at each step.
  - `min_samples_split`: Values from 50 to 150, incrementing by 30 at each step.

In [ ]:
# CV results

result = pd.DataFrame(grid_search.cv_results_)
result.head(3)

In [ ]:
# print the optimal accuracy score and hyperparameters

print("best accuracy", grid_search.best_score_)
print(grid_search.best_estimator_)

- `grid_search.best_score_`: the best achieved accuracy from the grid search.
- `grid_search.best_estimator_`: the best estimator obtained.

<hr>

### Run the model with best parameters obtained from grid search.
- The optimal parameter values might be different

In [ ]:
# model with optimal hyperparameters

m = DecisionTreeClassifier(criterion='entropy', max_depth=4, min_samples_leaf=75, min_samples_split=50)
m.fit(xtrain, ytrain)

In [ ]:
# accuracy

m.score(xtest, ytest)

In [ ]:
# Predict the target in the test data and display accuracy, confusion matrix

pred = m.predict(xtest)
print(confusion_matrix(ytest,pred))

In [ ]:
# Classification report

print(classification_report(ytest, pred))

In [ ]:
# Export the decision tree to DOT format
dot_data = export_graphviz(m, out_file=None, feature_names=xtrain.columns, class_names=["No", "Yes"],
                      filled=True, rounded=True, special_characters=True)

# Convert the DOT file to a graph
graph = pydotplus.graph_from_dot_data(dot_data)

# Display the decision tree as an image
Image(graph.create_png())